In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
# load data
df = pd.read_csv('../data/cleaned/wednesday_cleaned.csv')
print(df.shape)
df.head()

(61001, 69)


,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Attack
0,443,87261,1,1,0,0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
1,443,523,2,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
2,53,94304,1,1,56,338,56,56,56.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
3,53,207,2,2,84,246,42,42,42.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
4,123,69022032,2,2,96,96,48,48,48.0,0.0,...,22276.0,0.0,22276,22276,69000000.0,0.0,69000000,69000000,BENIGN,0


In [ ]:
# check what columns we have for sanity check
print(df.columns)

Index(['Destination_Port', 'Flow_Duration', 'Total_Fwd_Packets',
       'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets',
       'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max',
       'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean',
       'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max',
       'Bwd_Packet_Length_Min', 'Bwd_Packet_Length_Mean',
       'Bwd_Packet_Length_Std', 'Flow_Bytes/s', 'Flow_Packets/s',
       'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min',
       'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max',
       'Fwd_IAT_Min', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std',
       'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Fwd_Header_Length',
       'Bwd_Header_Length', 'Fwd_Packets/s', 'Bwd_Packets/s',
       'Min_Packet_Length', 'Max_Packet_Length', 'Packet_Length_Mean',
       'Packet_Length_Std', 'Packet_Length_Variance', 'FIN_Flag_Count',
       'SYN_Flag_Count', 'RST_Flag_Count', 'PSH_Flag_Count', 'ACK_Flag_Count'

In [ ]:
# get numeric ones only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(len(numeric_cols))

# dont need label and attack
if 'Attack' in numeric_cols:
    numeric_cols.remove('Attack')
if 'Label' in numeric_cols:
    numeric_cols.remove('Label')
    
print(f'total: {len(numeric_cols)}')

In [ ]:
# compute correlation matrix
corr = df[numeric_cols].corr()
print(corr.shape)

In [ ]:
# look at sample using head 
corr.head()

In [ ]:
# make the heatmap
plt.figure(figsize=(18, 16))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../eda/figures/bivariate/correlation_heatmap_full.png', dpi=300)
plt.show()

In [ ]:
# find pairs with high correlation
# gonna use 0.8 as the cutoff

pairs = []

for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) >= 0.8:
            pairs.append([corr.columns[i], corr.columns[j], corr.iloc[i, j]])

print(f'found {len(pairs)} pairs')

In [ ]:
# convert to df
df_pairs = pd.DataFrame(pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])

# sort it
df_pairs['abs'] = df_pairs['Correlation'].abs()
df_pairs = df_pairs.sort_values('abs', ascending=False)
df_pairs = df_pairs.drop('abs', axis=1)

print(df_pairs)

In [ ]:
# also check moderate ones (0.6-0.8)
moderate = []

for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = abs(corr.iloc[i, j])
        if val >= 0.6 and val < 0.8:
            moderate.append([corr.columns[i], corr.columns[j], corr.iloc[i, j]])

print(f'moderate pairs: {len(moderate)}')

In [ ]:
df_moderate = pd.DataFrame(moderate, columns=['Feature 1', 'Feature 2', 'Correlation'])
df_moderate['abs'] = df_moderate['Correlation'].abs()
df_moderate = df_moderate.sort_values('abs', ascending=False)
df_moderate = df_moderate.drop('abs', axis=1)

# just show top 20
print(df_moderate.head(20))

In [ ]:
# get all the correlation values to see overall distribution
all_vals = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        all_vals.append(corr.iloc[i, j])

print(f'total: {len(all_vals)}')
print(f'mean: {np.mean(all_vals)}')
print(f'max: {np.max(all_vals)}')
print(f'min: {np.min(all_vals)}')

In [ ]:
# count how many in different buckets
count_90 = 0
count_80 = 0
count_70 = 0
count_60 = 0

for c in all_vals:
    if abs(c) >= 0.9:
        count_90 += 1
    if abs(c) >= 0.8:
        count_80 += 1
    if abs(c) >= 0.7:
        count_70 += 1
    if abs(c) >= 0.6:
        count_60 += 1
        
print('corr >= 0.9:', count_90)
print('corr >= 0.8:', count_80)
print('corr >= 0.7:', count_70)
print('corr >= 0.6:', count_60)

In [ ]:
# plot histogram
plt.figure(figsize=(10, 6))
plt.hist(all_vals, bins=50, color='steelblue', edgecolor='black')
plt.xlabel('Correlation')
plt.ylabel('Frequency')
plt.title('Distribution of Correlations')
plt.axvline(x=0, color='red', linestyle='--', label='zero')
plt.axvline(x=0.8, color='orange', linestyle='--', label='threshold 0.8')
plt.axvline(x=-0.8, color='orange', linestyle='--')
plt.legend()
plt.savefig('../eda/figures/bivariate/correlation_distribution.png', dpi=300)
plt.show()

12/2/2025 - Umar

computed correlation matrix and made heatmap. found 131 pairs with correlation over 0.8 which is a lot. these are probably redundant and we should remove some before modeling. also made histogram showing most correlations are near zero but there are clusters of high ones.

next: probably need to pick which features to drop from the highly correlated pairs

### Done with Task